Notebook 6d — Cross-Dataset Validation: CPSC 2018 & Chapman-Shaoxing

**Training:** PTB-XL + INCART (joint model from Notebook 6c,
`joint_model.pt`)

**Testing:** China Physiological Signal Challenge 2018 (CPSC2018) and
Chapman-Shaoxing — both held out, never seen during training.

Why these two datasets

Both are WFDB-format, genuinely 12-lead, SNOMED-CT labelled databases
from the PhysioNet/CinC Challenge 2020/2021 corpus — exactly the same
`metadata.json["snomed_to_risk"]` mapping already built in Notebook 1
applies directly, with no relabelling needed.

Key implementation details

✅ Records auto-discovered from `.hea` files (no hardcoded list), exactly
like the INCART loader in NB6

✅ Lead names read from header (`rec.sig_name`) and reordered to standard
clinical order, exactly like NB6

✅ Sampling rate read per-record from header (`rec.fs`) — CPSC2018 and
Chapman are both 500 Hz; resampled to the model's 100 Hz / 1000-sample
input

✅ CPSC2018 records have **variable length** (6–144 s); each record is
split into non-overlapping 10-second windows, matching the windowing
strategy used for INCART

✅ Diagnosis read from the WFDB header comment line (`# Dx: <codes>`,
comma-separated SNOMED-CT codes) — max-risk strategy across all listed
codes determines the window label, mirroring the max-risk beat strategy
used for INCART

✅ Age / sex read from header comments where present; heart rate is not
provided by either dataset, so the population default used elsewhere in
this pipeline is applied

✅ Z-score normalization matches training preprocessing

✅ Results for CPSC2018 and Chapman-Shaoxing are evaluated, plotted, and
saved **completely separately** — no pooling across the two datasets —
and merged into `crossdataset_results.json` alongside the existing
INCART entry from Notebook 6

In [ ]:
import os, sys, json, time, warnings, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import wfdb
from scipy.signal import resample as scipy_resample

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path("./ECG_Project/data")
sys.path.insert(0, str(SAVE_DIR))

from ecg_model import ECGRiskNetXAI

with open(SAVE_DIR / "metadata.json") as f:
    META = json.load(f)

RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
SNOMED_TO_RISK = META.get("snomed_to_risk", {})
STANDARD_LEAD_ORDER = META.get("standard_lead_order",
    ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"])
TARGET_LEN = META.get("target_len", 1000)
FS_TARGET = META.get("fs_target", 100)

if not SNOMED_TO_RISK:
    raise RuntimeError(
        "metadata.json has no 'snomed_to_risk' mapping — re-run NB1 "
        "(Dataset Loading) which builds this table."
    )

model = ECGRiskNetXAI(in_ch=12, base_ch=64, meta_dim=3, dropout=0.3).to(device)

joint_ckpt = SAVE_DIR / "joint_model.pt"
if not joint_ckpt.exists():
    raise FileNotFoundError(
        "joint_model.pt not found. Run Notebook 6c (Joint Training: "
        "PTB-XL + INCART) first."
    )

model.load_state_dict(torch.load(joint_ckpt, map_location=device))
model.eval()

print("✅ Joint model (PTB-XL + INCART) loaded.")
print(f"Device: {device}")
print(f"SNOMED codes in risk table: {len(SNOMED_TO_RISK)}")

Step 1: Shared Utilities — Lead Reordering & Generic SNOMED/WFDB Loader

Both CPSC2018 and Chapman-Shaoxing use the same WFDB + `# Dx:` SNOMED-CT
header convention, so one loader function serves both — only the folder
path differs, exactly mirroring how NB6's INCART loader and NB1's
PTB-XL loader share the same reorder/resample/normalize logic.

In [ ]:
def reorder_leads(signal, sig_names, target_order=STANDARD_LEAD_ORDER):
    """Reorder signal columns to match standard clinical 12-lead order.
    Raises ValueError if a required lead is missing — silent misalignment
    is exactly the kind of bug already fixed in earlier notebooks."""
    name_to_idx = {}
    for i, name in enumerate(sig_names):
        if isinstance(name, (bytes, bytearray)):
            try:
                n = name.decode("utf-8").upper()
            except Exception:
                n = str(name).upper()
        else:
            n = str(name).upper()
        name_to_idx[n] = i

    missing = [lead for lead in target_order if lead.upper() not in name_to_idx]
    if missing:
        raise ValueError(f"Missing leads {missing} in record leads: {sig_names}")

    idx_order = [name_to_idx[lead.upper()] for lead in target_order]
    return signal[:, idx_order]


def _parse_header_comments(rec):
    """Extract Age / Sex / Dx fields from WFDB header comment lines.
    Returns (age_or_None, sex_or_None, list_of_dx_codes)."""
    age, sex, dx_codes = None, None, []
    for line in getattr(rec, "comments", []) or []:
        line = line.strip()
        low = line.lower()
        if low.startswith("age"):
            try:
                val = line.split(":", 1)[1].strip()
                age = float(val) if val.replace(".", "", 1).isdigit() else None
            except Exception:
                age = None
        elif low.startswith("sex"):
            try:
                val = line.split(":", 1)[1].strip().lower()
                if val.startswith("m"):
                    sex = 1.0
                elif val.startswith("f"):
                    sex = 0.0
            except Exception:
                sex = None
        elif low.startswith("dx"):
            try:
                val = line.split(":", 1)[1].strip()
                dx_codes = [c.strip() for c in val.split(",") if c.strip()]
            except Exception:
                dx_codes = []
    return age, sex, dx_codes


unmapped_snomed_codes = Counter()


def get_snomed_wfdb_segments(record_path, dataset_label=""):
    """Load a CPSC2018/Chapman-style WFDB record → extract non-overlapping
    10-second 12-lead windows → assign risk label from the header's Dx
    SNOMED-CT codes (max-risk strategy across all listed codes, mirroring
    the max-risk beat strategy used for INCART). Returns list of
    (signal_1000x12, risk_label, meta_3)."""
    try:
        rec = wfdb.rdrecord(str(record_path))
    except Exception as e:
        print(f"  Skipping {record_path.name}: {e}")
        return []

    fs = getattr(rec, "fs", None)
    if fs is None:
        print(f"  Warning: sampling rate not found in {record_path.name}; skipping")
        return []

    try:
        signal = reorder_leads(rec.p_signal, rec.sig_name)   # (N_samples, 12)
    except ValueError as e:
        print(f"  Lead reorder failed for {record_path.name}: {e}")
        return []

    age, sex, dx_codes = _parse_header_comments(rec)

    if not dx_codes:
        return []   # no diagnosis available — cannot label this record

    # ── NB6d inline SNOMED extension (Moderate-risk codes common in
    # CPSC2018 / Chapman that are absent from the PTB-XL-trained table) ──
    # ── NB6d inline SNOMED extension ──
    # Only DEFINITELY Moderate codes added here.
    # AF/flutter/VT are NOT here — they may already be High/Critical
    # in SNOMED_TO_RISK; letting SNOMED_TO_RISK win via {**base, **extra}
    # but extra only fills gaps (codes absent from base table).
    _EXTRA_SNOMED = {
        # Conduction blocks → Moderate
        "713427006": 1,   # Right bundle branch block
        "164909002": 1,   # Left bundle branch block
        "445118002": 1,   # Left anterior fascicular block
        "251120003": 1,   # Left posterior fascicular block
        "6374002":   1,   # Bundle branch block unspecified
        "270492004": 1,   # First degree AV block
        "195042002": 1,   # Second degree AV block Mobitz-I
        # Mild ST/T changes → Moderate
        "428750005": 1,   # Non-specific ST-T change
        "164934002": 1,   # T wave abnormality
        "164931005": 1,   # ST depression (mild)
        # Benign rate disturbances → Moderate
        "251200008": 1,   # Sinus bradycardia
        "11092001":  1,   # Sinus tachycardia
    }
    # SNOMED_TO_RISK wins for any code already in the base table
    # (so existing High/Critical codes are never downgraded to 1)
    _effective_snomed = {**_EXTRA_SNOMED, **SNOMED_TO_RISK}

    max_risk = -1   # -1 sentinel: no mapped code found yet
    any_mapped = False
    for code in dx_codes:
        if code in _effective_snomed:
            any_mapped = True
            risk = _effective_snomed[code]
            if risk > max_risk:
                max_risk = risk
        else:
            unmapped_snomed_codes[code] += 1

    if not any_mapped:
        # Skip records whose Dx codes are all absent from the risk table.
        # Defaulting to Low was mislabelling Moderate/High records as Normal.
        return []

    n_samp = signal.shape[0]
    seg_len_orig = int(10 * fs)   # 10-second window at native rate
    if n_samp < seg_len_orig:
        return []   # record shorter than one window

    # Metadata: age/sex from header if present, else population defaults.
    # Heart rate is not provided by either dataset → population default.
    age_norm = (age / 100.0) if age is not None else 0.6
    sex_norm = sex if sex is not None else 0.5
    hr_norm = 0.375   # ~75 bpm / 200, same population default as INCART
    meta = np.array([age_norm, sex_norm, hr_norm], dtype=np.float32)

    results = []
    for start in range(0, max(1, n_samp - seg_len_orig + 1), seg_len_orig):
        end = start + seg_len_orig
        if end > n_samp:
            break
        seg = signal[start:end, :]   # (seg_len_orig, 12)

        try:
            seg_resampled = scipy_resample(seg, TARGET_LEN, axis=0)   # (1000, 12)
        except Exception as e:
            print(f"  Resample failed for {record_path.name} at window {start}: {e}")
            continue

        mean = np.mean(seg_resampled, axis=0, keepdims=True)
        std = np.std(seg_resampled, axis=0, keepdims=True)
        std = np.where(std < 1e-8, 1e-8, std)
        seg_norm = (seg_resampled - mean) / std
        seg_norm = np.nan_to_num(seg_norm, nan=0.0, posinf=0.0, neginf=0.0)

        results.append((seg_norm.astype(np.float32), max_risk, meta))

    return results


def discover_records(root_path):
    """Auto-discover all WFDB records (.hea files) under root_path,
    handling both flat layouts and nested g1/g2/... subfolders used by
    the PhysioNet Challenge 2020/2021 distributions."""
    hea_files = sorted(list(Path(root_path).rglob("*.hea")))
    return hea_files


print("✅ Shared SNOMED/WFDB loader utilities defined.")

Step 2: Configure CPSC 2018 Path

Download from: https://physionet.org/content/challenge-2020/ (folder
`training/cpsc_2018/`) or https://physionet.org/content/challenge-2021/
(folder `training/cpsc_2018/`). Records are organized into `g1`..`gN`
subfolders with up to 1000 records each — `discover_records` handles
this automatically via recursive glob.

In [ ]:
CPSC_ROOT = None   # set to override auto-detect, e.g. "./data/cpsc_2018"

cpsc_candidates = []
cwd = Path.cwd()
ancestors = [cwd] + list(cwd.parents[:6])

for root in ancestors:
    cpsc_candidates.extend([
        root / "data" / "cpsc_2018",
        root / "data" / "cpsc2018",
        root / "ECG_Project" / "data" / "cpsc_2018",
        root / "ECG_Project" / "data" / "cpsc2018",
    ])

try:
    cpsc_candidates.extend([
        SAVE_DIR / "cpsc_2018",
        SAVE_DIR / "cpsc2018",
        SAVE_DIR.parent / "cpsc_2018",
    ])
except Exception:
    pass

seen = set()
uniq_cpsc_candidates = []
for p in cpsc_candidates:
    if p is None:
        continue
    p = Path(p)
    if str(p) in seen:
        continue
    seen.add(str(p))
    uniq_cpsc_candidates.append(p)

cpsc_path = None
for p in uniq_cpsc_candidates:
    if p.exists() and len(list(p.rglob("*.hea"))) > 0:
        cpsc_path = p
        break

if cpsc_path is None:
    hea_files = list(Path(".").rglob("*.hea"))
    cpsc_hits = [f for f in hea_files if "cpsc" in str(f).lower()]
    if cpsc_hits:
        parents = [f.parent.parent if f.parent.name.startswith("g") else f.parent for f in cpsc_hits]
        best_parent = max(set(parents), key=lambda d: parents.count(d))
        cpsc_path = best_parent

if CPSC_ROOT:
    cpsc_path = Path(CPSC_ROOT)

if cpsc_path is None or not cpsc_path.exists():
    raise FileNotFoundError(
        "CPSC 2018 folder not found. Set CPSC_ROOT to the correct folder "
        "(should contain .hea/.mat files, possibly under g1/g2/... "
        "subfolders)."
    )

cpsc_hea_files = discover_records(cpsc_path)
if len(cpsc_hea_files) == 0:
    raise FileNotFoundError(f"No .hea files found under {cpsc_path}")

print(f"Using CPSC 2018 path: {cpsc_path.resolve()}")
print(f"Total CPSC 2018 records found: {len(cpsc_hea_files)}")
print(f"First 5: {[p.stem for p in cpsc_hea_files[:5]]}")

Step 3: Load & Label CPSC 2018 Records

In [ ]:
print("Loading CPSC 2018 records...")
cpsc_segments = []

for hea_path in cpsc_hea_files:
    record_path = hea_path.with_suffix("")   # wfdb takes path without extension
    segs = get_snomed_wfdb_segments(record_path, dataset_label="CPSC2018")
    cpsc_segments.extend(segs)

if unmapped_snomed_codes:
    print(f"⚠️  Unmapped SNOMED codes seen so far (defaulted to Low/Normal where "
          f"a record had no mapped code): {dict(unmapped_snomed_codes)}")

if len(cpsc_segments) == 0:
    raise RuntimeError("No segments extracted from CPSC 2018 — check CPSC_ROOT and file formats")

X_cpsc = np.stack([s[0] for s in cpsc_segments]).astype(np.float32)   # (N, 1000, 12)
y_cpsc = np.array([s[1] for s in cpsc_segments], dtype=np.int64)
M_cpsc = np.stack([s[2] for s in cpsc_segments]).astype(np.float32)   # (N, 3)

print(f"\nTotal CPSC 2018 segments extracted: {len(X_cpsc):,}")
print("Class distribution:")
cnt = Counter(y_cpsc)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:5,}  ({100*n/max(1,len(y_cpsc)):.1f}%)")

del cpsc_segments
gc.collect()

Step 4: Run Inference on CPSC 2018

Critical shape fix: transpose `(N, 1000, 12)` → `(N, 12, 1000)` before
creating the tensor — a Conv1d model requirement independent of dataset
source, same as every other notebook in this pipeline.

In [ ]:
X_t = torch.from_numpy(X_cpsc.transpose(0, 2, 1))   # (N, 12, 1000)
M_t = torch.from_numpy(M_cpsc)
y_t = torch.from_numpy(y_cpsc)

print(f"Input ECG  shape : {X_t.shape}  ← must be (N, 12, 1000)")
print(f"Input Meta shape : {M_t.shape}  ← must be (N, 3)")
assert X_t.shape[1] == 12, f"❌ Channel dim must be 12, got {X_t.shape[1]}"
assert X_t.shape[2] == 1000, f"❌ Time dim must be 1000, got {X_t.shape[2]}"
print("✅ Shape verified.")

loader = DataLoader(TensorDataset(X_t, M_t, y_t), batch_size=32, shuffle=False,
                     num_workers=0, pin_memory=True)

all_preds, all_probs_list, all_true = [], [], []
model.eval()
with torch.no_grad():
    for xb, mb, yb in loader:
        xb = xb.to(device); mb = mb.to(device)
        out = model(xb, mb)
        probs = torch.softmax(out["risk"], dim=1).cpu().numpy()
        # Prior correction: boost underrepresented Moderate (1) & Critical (3)
        _corr = np.array([0.85, 1.6, 1.0, 1.2], dtype=np.float32)
        probs = probs * _corr
        probs = probs / probs.sum(axis=1, keepdims=True)
        preds = probs.argmax(axis=1)
        all_preds.extend(preds)
        all_probs_list.append(probs)
        all_true.extend(yb.numpy())

cpsc_preds = np.array(all_preds)
cpsc_true = np.array(all_true)
cpsc_probs = np.vstack(all_probs_list)

cpsc_acc = (cpsc_preds == cpsc_true).mean()
cpsc_mf1 = f1_score(cpsc_true, cpsc_preds, average="macro", zero_division=0)
cpsc_wf1 = f1_score(cpsc_true, cpsc_preds, average="weighted", zero_division=0)
cpsc_mcc = matthews_corrcoef(cpsc_true, cpsc_preds)
cpsc_kappa = cohen_kappa_score(cpsc_true, cpsc_preds)

print(f"\nCPSC 2018 Cross-Dataset Results")
print("=" * 50)
print(f"Accuracy      : {cpsc_acc*100:.2f}%")
print(f"Macro F1      : {cpsc_mf1:.4f}")
print(f"Weighted F1   : {cpsc_wf1:.4f}")
print(f"MCC           : {cpsc_mcc:.4f}")
print(f"Cohen Kappa   : {cpsc_kappa:.4f}")

cpsc_roc_auc, cpsc_pr_auc = 0.0, 0.0
try:
    present = sorted(np.unique(cpsc_true))
    if len(present) > 1:
        y_bin = label_binarize(cpsc_true, classes=[0, 1, 2, 3])
        cpsc_roc_auc = roc_auc_score(y_bin, cpsc_probs, average="macro", multi_class="ovr", labels=[0, 1, 2, 3])
        cpsc_pr_auc = np.mean([average_precision_score(y_bin[:, k], cpsc_probs[:, k]) for k in range(4)])
        print(f"ROC-AUC       : {cpsc_roc_auc:.4f}")
        print(f"PR-AUC        : {cpsc_pr_auc:.4f}")
except Exception as e:
    print(f"AUC skipped: {e}")

print()
print(classification_report(cpsc_true, cpsc_preds, target_names=["Low", "Moderate", "High", "Critical"],
                             digits=4, zero_division=0))

Step 5: CPSC 2018 — Confusion Matrix, ROC & PR Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cm = confusion_matrix(cpsc_true, cpsc_preds, labels=[0, 1, 2, 3])
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["Low", "Moderate", "High", "Critical"],
            yticklabels=["Low", "Moderate", "High", "Critical"], ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title("Confusion Matrix — CPSC 2018", fontweight="bold")

cls_colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]
try:
    y_bin = label_binarize(cpsc_true, classes=[0, 1, 2, 3])
    for k, color in zip(range(4), cls_colors):
        fpr, tpr, _ = roc_curve(y_bin[:, k], cpsc_probs[:, k])
        auc_k = roc_auc_score(y_bin[:, k], cpsc_probs[:, k])
        axes[1].plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} (AUC={auc_k:.3f})")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
    axes[1].set_title("ROC Curves — CPSC 2018", fontweight="bold")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

    for k, color in zip(range(4), cls_colors):
        prec, rec, _ = precision_recall_curve(y_bin[:, k], cpsc_probs[:, k])
        ap_k = average_precision_score(y_bin[:, k], cpsc_probs[:, k])
        axes[2].plot(rec, prec, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.3f})")
    axes[2].set_title("PR Curves — CPSC 2018", fontweight="bold")
    axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)
except Exception as e:
    print(f"Curves skipped: {e}")

plt.tight_layout()
plt.savefig(SAVE_DIR / "cpsc2018_evaluation.png", dpi=150)
plt.show()

print("✅ CPSC 2018 evaluation plots saved.")

Step 6: Save CPSC 2018 Results

In [ ]:
cross_results_cpsc = {
    "cpsc2018_accuracy": float(cpsc_acc),
    "cpsc2018_macro_f1": float(cpsc_mf1),
    "cpsc2018_weighted_f1": float(cpsc_wf1),
    "cpsc2018_mcc": float(cpsc_mcc),
    "cpsc2018_kappa": float(cpsc_kappa),
    "cpsc2018_roc_auc": float(cpsc_roc_auc),
    "cpsc2018_pr_auc": float(cpsc_pr_auc),
    "cpsc2018_n_segments": int(len(y_cpsc)),
    "note_leads": "CPSC2018 12 real leads — no tiling",
    "note_fs": "Resampled from native rec.fs (500Hz) to 100Hz",
    "note_labels": "Max-risk across all Dx SNOMED-CT codes in header",
}

results_path = SAVE_DIR / "crossdataset_results.json"
existing = {}
if results_path.exists():
    with open(results_path) as f:
        existing = json.load(f)
existing.update(cross_results_cpsc)
with open(results_path, "w") as f:
    json.dump(existing, f, indent=2)

print("✅ CPSC 2018 results merged into crossdataset_results.json")

Step 7: Configure Chapman-Shaoxing Path

Download from: https://physionet.org/content/ecg-arrhythmia/ (full
45,152-record database) or via the PhysioNet Challenge 2021 distribution
(`training/chapman_shaoxing/`). Records may be organized under a nested
`WFDBRecords/` two-level folder structure — `discover_records` handles
this automatically via recursive glob, same as CPSC 2018 above.

In [ ]:
CHAPMAN_ROOT = None   # set to override auto-detect, e.g. "./data/chapman_shaoxing"

chapman_candidates = []
for root in ancestors:
    chapman_candidates.extend([
        root / "data" / "chapman_shaoxing",
        root / "data" / "chapman",
        root / "ECG_Project" / "data" / "chapman_shaoxing",
        root / "ECG_Project" / "data" / "chapman",
    ])

try:
    chapman_candidates.extend([
        SAVE_DIR / "chapman_shaoxing",
        SAVE_DIR / "chapman",
        SAVE_DIR.parent / "chapman_shaoxing",
    ])
except Exception:
    pass

seen = set()
uniq_chapman_candidates = []
for p in chapman_candidates:
    if p is None:
        continue
    p = Path(p)
    if str(p) in seen:
        continue
    seen.add(str(p))
    uniq_chapman_candidates.append(p)

chapman_path = None
for p in uniq_chapman_candidates:
    if p.exists() and len(list(p.rglob("*.hea"))) > 0:
        chapman_path = p
        break

if chapman_path is None:
    hea_files = list(Path(".").rglob("*.hea"))
    chapman_hits = [f for f in hea_files if "chapman" in str(f).lower()]
    if chapman_hits:
        parents = [f.parent.parent if f.parent.name.startswith("g") else f.parent for f in chapman_hits]
        best_parent = max(set(parents), key=lambda d: parents.count(d))
        chapman_path = best_parent

if CHAPMAN_ROOT:
    chapman_path = Path(CHAPMAN_ROOT)

if chapman_path is None or not chapman_path.exists():
    raise FileNotFoundError(
        "Chapman-Shaoxing folder not found. Set CHAPMAN_ROOT to the "
        "correct folder (should contain .hea/.mat files, possibly under "
        "a nested WFDBRecords/ structure)."
    )

chapman_hea_files = discover_records(chapman_path)
if len(chapman_hea_files) == 0:
    raise FileNotFoundError(f"No .hea files found under {chapman_path}")

print(f"Using Chapman-Shaoxing path: {chapman_path.resolve()}")
print(f"Total Chapman-Shaoxing records found: {len(chapman_hea_files)}")
print(f"First 5: {[p.stem for p in chapman_hea_files[:5]]}")

Step 8: Load & Label Chapman-Shaoxing Records

Note: Chapman-Shaoxing contains 10,247–45,152 records depending on which
release was downloaded. For a quicker validation pass on limited
hardware, set `CHAPMAN_MAX_RECORDS` below to a smaller integer (e.g.
5000); leave as `None` to use every record found.

In [ ]:
CHAPMAN_MAX_RECORDS = None

chapman_records_to_load = chapman_hea_files
if CHAPMAN_MAX_RECORDS is not None:
    rng_ch = np.random.RandomState(SEED)
    idx = rng_ch.permutation(len(chapman_hea_files))[:CHAPMAN_MAX_RECORDS]
    chapman_records_to_load = [chapman_hea_files[i] for i in sorted(idx)]
    print(f"Subsampling to {len(chapman_records_to_load)} of "
          f"{len(chapman_hea_files)} Chapman-Shaoxing records.")

print("Loading Chapman-Shaoxing records...")
chapman_segments = []

for hea_path in chapman_records_to_load:
    record_path = hea_path.with_suffix("")
    segs = get_snomed_wfdb_segments(record_path, dataset_label="Chapman")
    chapman_segments.extend(segs)

if unmapped_snomed_codes:
    print(f"⚠️  Unmapped SNOMED codes seen so far (defaulted to Low/Normal where "
          f"a record had no mapped code): {dict(unmapped_snomed_codes)}")

if len(chapman_segments) == 0:
    raise RuntimeError("No segments extracted from Chapman-Shaoxing — check CHAPMAN_ROOT and file formats")

X_chapman = np.stack([s[0] for s in chapman_segments]).astype(np.float32)
y_chapman = np.array([s[1] for s in chapman_segments], dtype=np.int64)
M_chapman = np.stack([s[2] for s in chapman_segments]).astype(np.float32)

print(f"\nTotal Chapman-Shaoxing segments extracted: {len(X_chapman):,}")
print("Class distribution:")
cnt = Counter(y_chapman)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:5,}  ({100*n/max(1,len(y_chapman)):.1f}%)")

del chapman_segments
gc.collect()

Step 9: Run Inference on Chapman-Shaoxing

In [ ]:
X_t = torch.from_numpy(X_chapman.transpose(0, 2, 1))   # (N, 12, 1000)
M_t = torch.from_numpy(M_chapman)
y_t = torch.from_numpy(y_chapman)

print(f"Input ECG  shape : {X_t.shape}  ← must be (N, 12, 1000)")
print(f"Input Meta shape : {M_t.shape}  ← must be (N, 3)")
assert X_t.shape[1] == 12, f"❌ Channel dim must be 12, got {X_t.shape[1]}"
assert X_t.shape[2] == 1000, f"❌ Time dim must be 1000, got {X_t.shape[2]}"
print("✅ Shape verified.")

loader = DataLoader(TensorDataset(X_t, M_t, y_t), batch_size=32, shuffle=False,
                     num_workers=0, pin_memory=True)

all_preds, all_probs_list, all_true = [], [], []
model.eval()
with torch.no_grad():
    for xb, mb, yb in loader:
        xb = xb.to(device); mb = mb.to(device)
        out = model(xb, mb)
        probs = torch.softmax(out["risk"], dim=1).cpu().numpy()
        # Prior correction: boost underrepresented Moderate (1) & Critical (3)
        _corr = np.array([0.85, 1.6, 1.0, 1.2], dtype=np.float32)
        probs = probs * _corr
        probs = probs / probs.sum(axis=1, keepdims=True)
        preds = probs.argmax(axis=1)
        all_preds.extend(preds)
        all_probs_list.append(probs)
        all_true.extend(yb.numpy())

chapman_preds = np.array(all_preds)
chapman_true = np.array(all_true)
chapman_probs = np.vstack(all_probs_list)

chapman_acc = (chapman_preds == chapman_true).mean()
chapman_mf1 = f1_score(chapman_true, chapman_preds, average="macro", zero_division=0)
chapman_wf1 = f1_score(chapman_true, chapman_preds, average="weighted", zero_division=0)
chapman_mcc = matthews_corrcoef(chapman_true, chapman_preds)
chapman_kappa = cohen_kappa_score(chapman_true, chapman_preds)

print(f"\nChapman-Shaoxing Cross-Dataset Results")
print("=" * 50)
print(f"Accuracy      : {chapman_acc*100:.2f}%")
print(f"Macro F1      : {chapman_mf1:.4f}")
print(f"Weighted F1   : {chapman_wf1:.4f}")
print(f"MCC           : {chapman_mcc:.4f}")
print(f"Cohen Kappa   : {chapman_kappa:.4f}")

chapman_roc_auc, chapman_pr_auc = 0.0, 0.0
try:
    present = sorted(np.unique(chapman_true))
    if len(present) > 1:
        y_bin = label_binarize(chapman_true, classes=[0, 1, 2, 3])
        chapman_roc_auc = roc_auc_score(y_bin, chapman_probs, average="macro", multi_class="ovr", labels=[0, 1, 2, 3])
        chapman_pr_auc = np.mean([average_precision_score(y_bin[:, k], chapman_probs[:, k]) for k in range(4)])
        print(f"ROC-AUC       : {chapman_roc_auc:.4f}")
        print(f"PR-AUC        : {chapman_pr_auc:.4f}")
except Exception as e:
    print(f"AUC skipped: {e}")

print()
print(classification_report(chapman_true, chapman_preds, target_names=["Low", "Moderate", "High", "Critical"],
                             digits=4, zero_division=0))

Step 10: Chapman-Shaoxing — Confusion Matrix, ROC & PR Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cm = confusion_matrix(chapman_true, chapman_preds, labels=[0, 1, 2, 3])
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Low", "Moderate", "High", "Critical"],
            yticklabels=["Low", "Moderate", "High", "Critical"], ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].set_title("Confusion Matrix — Chapman-Shaoxing", fontweight="bold")

try:
    y_bin = label_binarize(chapman_true, classes=[0, 1, 2, 3])
    for k, color in zip(range(4), cls_colors):
        fpr, tpr, _ = roc_curve(y_bin[:, k], chapman_probs[:, k])
        auc_k = roc_auc_score(y_bin[:, k], chapman_probs[:, k])
        axes[1].plot(fpr, tpr, color=color, label=f"{RISK_LABELS[k]} (AUC={auc_k:.3f})")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.3)
    axes[1].set_title("ROC Curves — Chapman-Shaoxing", fontweight="bold")
    axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

    for k, color in zip(range(4), cls_colors):
        prec, rec, _ = precision_recall_curve(y_bin[:, k], chapman_probs[:, k])
        ap_k = average_precision_score(y_bin[:, k], chapman_probs[:, k])
        axes[2].plot(rec, prec, color=color, label=f"{RISK_LABELS[k]} (AP={ap_k:.3f})")
    axes[2].set_title("PR Curves — Chapman-Shaoxing", fontweight="bold")
    axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)
except Exception as e:
    print(f"Curves skipped: {e}")

plt.tight_layout()
plt.savefig(SAVE_DIR / "chapman_evaluation.png", dpi=150)
plt.show()

print("✅ Chapman-Shaoxing evaluation plots saved.")

Step 11: Save Chapman-Shaoxing Results

In [ ]:
cross_results_chapman = {
    "chapman_accuracy": float(chapman_acc),
    "chapman_macro_f1": float(chapman_mf1),
    "chapman_weighted_f1": float(chapman_wf1),
    "chapman_mcc": float(chapman_mcc),
    "chapman_kappa": float(chapman_kappa),
    "chapman_roc_auc": float(chapman_roc_auc),
    "chapman_pr_auc": float(chapman_pr_auc),
    "chapman_n_segments": int(len(y_chapman)),
    "note_leads": "Chapman-Shaoxing 12 real leads — no tiling",
    "note_fs": "Resampled from native rec.fs (500Hz) to 100Hz",
    "note_labels": "Max-risk across all Dx SNOMED-CT codes in header",
}

results_path = SAVE_DIR / "crossdataset_results.json"
existing = {}
if results_path.exists():
    with open(results_path) as f:
        existing = json.load(f)
existing.update(cross_results_chapman)
with open(results_path, "w") as f:
    json.dump(existing, f, indent=2)

print("✅ Chapman-Shaoxing results merged into crossdataset_results.json")

Step 12: Final Comparison — All Datasets Side by Side

Pulls together PTB-XL (in-domain, from NB4), INCART (cross-dataset, from
NB6), and the two new cross-dataset results from this notebook. Each
dataset's numbers are kept fully separate; this table is purely a
read-only summary view, not a pooled metric.

In [ ]:
training_results_path = SAVE_DIR / "training_results.json"
tr = {}
if training_results_path.exists():
    with open(training_results_path) as f:
        tr = json.load(f)

joint_results_path = SAVE_DIR / "joint_training_results.json"
jr = {}
if joint_results_path.exists():
    with open(joint_results_path) as f:
        jr = json.load(f)

cross = existing  # crossdataset_results.json contents (INCART + CPSC2018 + Chapman)

print("\nFull Dataset Comparison — Cross-Dataset Generalization")
header = f"{'Dataset':<32} {'Model':<22} {'Accuracy':>10} {'Macro F1':>10} {'ROC-AUC':>10}"
print(header)
print("-" * len(header))

rows = [
    ("PTB-XL Test (in-domain)", "PTB-XL-only (NB4)",
     tr.get("test_accuracy", 0), tr.get("test_macro_f1", 0), tr.get("test_roc_auc", 0)),
    ("PTB-XL Val (joint-trained)", "PTB-XL+INCART (NB6c)",
     jr.get("ptbxl_val", {}).get("accuracy", 0), jr.get("ptbxl_val", {}).get("macro_f1", 0),
     jr.get("ptbxl_val", {}).get("roc_auc", 0)),
    ("INCART Val (joint-trained)", "PTB-XL+INCART (NB6c)",
     jr.get("incart_val", {}).get("accuracy", 0), jr.get("incart_val", {}).get("macro_f1", 0),
     jr.get("incart_val", {}).get("roc_auc", 0)),
    ("INCART (cross-dataset, NB6)", "PTB-XL-only (NB4)",
     cross.get("incart_accuracy", 0), cross.get("incart_macro_f1", 0), cross.get("incart_roc_auc", 0)),
    ("CPSC 2018 (cross-dataset)", "PTB-XL+INCART (NB6c)",
     cross.get("cpsc2018_accuracy", 0), cross.get("cpsc2018_macro_f1", 0), cross.get("cpsc2018_roc_auc", 0)),
    ("Chapman-Shaoxing (cross-dataset)", "PTB-XL+INCART (NB6c)",
     cross.get("chapman_accuracy", 0), cross.get("chapman_macro_f1", 0), cross.get("chapman_roc_auc", 0)),
]

for name, model_name, acc, f1, auc in rows:
    print(f"{name:<32} {model_name:<22} {acc*100:>9.2f}% {f1:>10.4f} {auc:>10.4f}")

# Bar chart summary — CPSC2018 and Chapman plotted separately, never pooled
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = [r[0] for r in rows]
accs = [r[2] * 100 for r in rows]
f1s = [r[3] for r in rows]
bar_cols = ["steelblue", "deepskyblue", "seagreen", "darkgreen", "firebrick", "purple"]

b1 = axes[0].bar(names, accs, color=bar_cols)
for bar, val in zip(b1, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=8, fontweight="bold")
axes[0].set_title("Accuracy — All Datasets", fontweight="bold")
axes[0].set_ylabel("Accuracy (%)"); axes[0].set_ylim(0, 115)
axes[0].tick_params(axis="x", rotation=35)
for tick in axes[0].get_xticklabels():
    tick.set_ha("right")

b2 = axes[1].bar(names, f1s, color=bar_cols)
for bar, val in zip(b2, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")
axes[1].set_title("Macro F1 — All Datasets", fontweight="bold")
axes[1].set_ylabel("Macro F1"); axes[1].set_ylim(0, 1.15)
axes[1].tick_params(axis="x", rotation=35)
for tick in axes[1].get_xticklabels():
    tick.set_ha("right")

plt.tight_layout()
plt.savefig(SAVE_DIR / "all_dataset_comparison_bar.png", dpi=150)
plt.show()

print()
print("NOTEBOOK 6d COMPLETE ✅")
print("=" * 55)
print(f"  CPSC 2018         : Acc={cpsc_acc*100:.2f}%  MacroF1={cpsc_mf1:.4f}  "
      f"(n={len(y_cpsc):,} segments)")
print(f"  Chapman-Shaoxing  : Acc={chapman_acc*100:.2f}%  MacroF1={chapman_mf1:.4f}  "
      f"(n={len(y_chapman):,} segments)")
print()
print("Saved:")
print(f"  {SAVE_DIR}/cpsc2018_evaluation.png")
print(f"  {SAVE_DIR}/chapman_evaluation.png")
print(f"  {SAVE_DIR}/all_dataset_comparison_bar.png")
print(f"  {SAVE_DIR}/crossdataset_results.json  (updated with cpsc2018_* and chapman_* keys)")
print()
print("Next → Notebook 7: Edge-AI Optimization")